# Diffusion-NullFusion — SOTA-beater for Chikusei x4

Multi-Scale Swin Transformer U-Net + DDPM noise prediction in nullspace + Sensor-Adaptive AdaLN conditioning.

Target: Beat CoFusion 50.67 / SMGU-Net 49.83 PSNR on Chikusei x4

In [ ]:
!pip install --quiet scipy h5py 2>/dev/null
import os
os.makedirs('/kaggle/working/diffusion_nullfusion', exist_ok=True)
print('Setup done')

In [ ]:
%%writefile train_diffusion_nullfusion.py
"""Diffusion-NullFusion — SOTA-beater for Chikusei x4."""
from __future__ import annotations
import argparse, glob, json, math, os, random, sys, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from scipy.io import loadmat
from scipy.ndimage import convolve, uniform_filter
import h5py
from torch.utils.data import Dataset

def chikusei_srf(bands=128):
    wl=np.linspace(363.0,1018.0,128)
    raw=np.stack([np.exp(-((wl-620)**2)/(2*80**2)),np.exp(-((wl-540)**2)/(2*70**2)),np.exp(-((wl-460)**2)/(2*60**2))],axis=1).astype(np.float32)
    return raw/np.maximum(raw.sum(axis=0,keepdims=True),1e-8)

def gaussian_kernel2d(size=9,sigma=1.2):
    ax=np.arange(size,dtype=np.float32)-(size-1)/2; xx,yy=np.meshgrid(ax,ax)
    k=np.exp(-0.5*(xx**2+yy**2)/sigma**2); return (k/k.sum()).astype(np.float32)

class DegradationOp(nn.Module):
    def __init__(self,scale,ksize=9,sigma=1.2):
        super().__init__(); self.scale=scale
        k=gaussian_kernel2d(ksize,sigma)
        self.register_buffer('k',torch.from_numpy(k)[None,None])
    def forward(self,x):
        C=x.shape[1]
        return F.conv2d(x,self.k.repeat(C,1,1,1),padding=4,groups=C)[:,:,::self.scale,::self.scale]
    def transpose(self,y,hw):
        C=y.shape[1]; H,W=hw
        up=torch.zeros(y.shape[0],C,H,W,device=y.device,dtype=y.dtype)
        up[:,:,::self.scale,::self.scale]=y
        return F.conv2d(up,self.k.repeat(C,1,1,1),padding=4,groups=C)

def scalar_cg(applyA,rhs,steps,tol=1e-10):
    z=torch.zeros_like(rhs); r=rhs-applyA(z); p=r.clone()
    rs=(r*r).flatten(1).sum(1)
    for _ in range(steps):
        ap=applyA(p); denom=(p*ap).flatten(1).sum(1)
        alpha=(rs/denom.clamp_min(tol)).reshape(-1,1,1,1)
        z=z+alpha*p; r=r-alpha*ap
        rs_new=(r*r).flatten(1).sum(1)
        beta=(rs_new/rs.clamp_min(tol)).reshape(-1,1,1,1)
        p=r+beta*p; rs=rs_new
    return z

class RangeNullProjector(nn.Module):
    def __init__(self,scale,cg_steps=40,ridge=1e-4):
        super().__init__(); self.D=DegradationOp(scale); self.scale=scale
        self.cg_steps=cg_steps; self.ridge=ridge
    def _normal_op(self,out_hw):
        def applyA(z): return self.D(self.D.transpose(z,out_hw))+self.ridge*z
        return applyA
    def pinv(self,yH,out_hw):
        return self.D.transpose(scalar_cg(self._normal_op(out_hw),yH,self.cg_steps),out_hw)
    def project_null(self,v,out_hw=None):
        if out_hw is None: out_hw=(v.shape[-2],v.shape[-1])
        return v-self.pinv(self.D(v),out_hw)

class WindowAttention(nn.Module):
    def __init__(self,dim,num_heads,window_size=8):
        super().__init__(); self.dim=dim; self.num_heads=num_heads
        self.head_dim=dim//num_heads; self.scale=self.head_dim**-0.5
        self.qkv=nn.Linear(dim,dim*3); self.proj=nn.Linear(dim,dim)
        self.window_size=window_size
    def forward(self,x):
        B,N,C=x.shape; H=W=int(math.sqrt(N)); ws=self.window_size
        x2d=x.reshape(B,H,W,C); pad_h=(ws-H%ws)%ws; pad_w=(ws-W%ws)%ws
        if pad_h or pad_w: x2d=F.pad(x2d,(0,0,0,pad_w,0,pad_h))
        Hp,Wp=H+pad_h,W+pad_w
        xw=x2d.reshape(B,Hp//ws,ws,Wp//ws,ws,C).permute(0,1,3,2,4,5).reshape(-1,ws*ws,C)
        qkv=self.qkv(xw).reshape(-1,3,self.num_heads,self.head_dim)
        q,k,v=qkv.unbind(1)
        attn=((q@k.transpose(-2,-1))*self.scale).softmax(dim=-1)
        out=(attn@v).transpose(1,2).reshape(-1,ws*ws,C)
        out=self.proj(out).reshape(B,Hp//ws,Wp//ws,ws,ws,C).permute(0,1,3,2,4,5).reshape(B,Hp,Wp,C)
        if pad_h or pad_w: out=out[:,:H,:W,:]
        return out.reshape(B,H*W,C)

class AdaLN(nn.Module):
    def __init__(self,dim,cond_dim):
        super().__init__(); self.norm=nn.LayerNorm(dim,elementwise_affine=False)
        self.proj=nn.Linear(cond_dim,dim*2)
    def forward(self,x,cond):
        gamma,beta=self.proj(cond).unsqueeze(1).chunk(2,dim=-1)
        return self.norm(x)*(1+gamma)+beta

class SpectralAttention(nn.Module):
    def __init__(self,dim,reduction=4):
        super().__init__()
        hidden=max(dim//reduction,8)
        self.fc=nn.Sequential(nn.Linear(dim,hidden),nn.GELU(),nn.Linear(hidden,dim),nn.Sigmoid())
    def forward(self,x):
        gate=self.fc(x.mean(dim=1))
        return x*gate.unsqueeze(1)

class SwinBlock(nn.Module):
    def __init__(self,dim,num_heads,cond_dim,window_size=8,mlp_ratio=4.0):
        super().__init__()
        self.norm1=AdaLN(dim,cond_dim); self.attn=WindowAttention(dim,num_heads,window_size)
        self.norm2=AdaLN(dim,cond_dim)
        self.mlp=nn.Sequential(nn.Linear(dim,int(dim*mlp_ratio)),nn.GELU(),nn.Linear(int(dim*mlp_ratio),dim))
        self.spec=SpectralAttention(dim)
    def forward(self,x,cond):
        x=x+self.attn(self.norm1(x,cond)); x=x+self.mlp(self.norm2(x,cond)); x=self.spec(x); return x

class PatchMerging(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.reduction=nn.Linear(4*dim,2*dim,bias=False); self.norm=nn.LayerNorm(4*dim)
    def forward(self,x,H,W):
        B=x.shape[0]; x=x.reshape(B,H,W,-1)
        x0,x1,x2,x3=x[:,0::2,0::2],x[:,1::2,0::2],x[:,0::2,1::2],x[:,1::2,1::2]
        x=torch.cat([x0,x1,x2,x3],-1).reshape(B,-1,4*x.shape[-1])
        return self.reduction(self.norm(x)),H//2,W//2

class PatchExpanding(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.linear=nn.Linear(dim,4*dim); self.norm=nn.LayerNorm(dim)
    def forward(self,x,H,W):
        x=self.linear(self.norm(x)).reshape(x.shape[0],H,W,2,2,-1).permute(0,1,3,2,4,5).reshape(x.shape[0],H*2,W*2,-1)
        return x.reshape(x.shape[0],H*2*W*2,-1),H*2,W*2

class ChannelProject(nn.Module):
    def __init__(self,in_dim,out_dim):
        super().__init__(); self.proj=nn.Linear(in_dim,out_dim)
    def forward(self,x): return self.proj(x)

class MultiScaleSwinUNet(nn.Module):
    def __init__(self,in_ch=128,base_dim=48,num_heads=None,cond_dim=64,window_size=8,depths=None,use_checkpoint=False):
        super().__init__()
        if num_heads is None: num_heads=[4,8,16]
        if depths is None: depths=[1,1,2,1,1]
        dims=[base_dim,base_dim*2,base_dim*4]
        self.use_checkpoint=use_checkpoint
        self.input_proj=nn.Linear(in_ch,dims[0])
        self.enc1=nn.ModuleList([SwinBlock(dims[0],num_heads[0],cond_dim,window_size) for _ in range(depths[0])])
        self.merge1=PatchMerging(dims[0])
        self.enc2=nn.ModuleList([SwinBlock(dims[1],num_heads[1],cond_dim,window_size) for _ in range(depths[1])])
        self.merge2=PatchMerging(dims[1])
        self.bottleneck=nn.ModuleList([SwinBlock(dims[2],num_heads[2],cond_dim,window_size) for _ in range(depths[2])])
        self.expand2=PatchExpanding(dims[2]); self.proj_skip2=ChannelProject(dims[1],dims[2]); self.proj_dec2=ChannelProject(dims[2],dims[1])
        self.dec2=nn.ModuleList([SwinBlock(dims[1],num_heads[1],cond_dim,window_size) for _ in range(depths[3])])
        self.expand1=PatchExpanding(dims[1]); self.proj_skip1=ChannelProject(dims[0],dims[1]); self.proj_dec1=ChannelProject(dims[1],dims[0])
        self.dec1=nn.ModuleList([SwinBlock(dims[0],num_heads[0],cond_dim,window_size) for _ in range(depths[4])])
        self.output_proj=nn.Linear(dims[0],in_ch)
    def _run(self,blocks,x,cond):
        for b in blocks:
            if self.use_checkpoint and self.training and x.requires_grad:
                x=checkpoint(b,x,cond,use_reentrant=False)
            else:
                x=b(x,cond)
        return x
    def forward(self,x,cond):
        B,C,H,W=x.shape; x=x.reshape(B,C,H*W).permute(0,2,1)
        x=self.input_proj(x)
        x=self._run(self.enc1,x,cond)
        skip1=x; x,H1,W1=self.merge1(x,H,W)
        x=self._run(self.enc2,x,cond)
        skip2=x; x,H2,W2=self.merge2(x,H1,W1)
        x=self._run(self.bottleneck,x,cond)
        x,H2,W2=self.expand2(x,H2,W2); x=self.proj_dec2(x+self.proj_skip2(skip2))
        x=self._run(self.dec2,x,cond)
        x,H1,W1=self.expand1(x,H1,W1); x=self.proj_dec1(x+self.proj_skip1(skip1))
        x=self._run(self.dec1,x,cond)
        return self.output_proj(x).reshape(B,H,W,-1).permute(0,3,1,2)

class CosineSchedule:
    def __init__(self,T=1000,s=0.008):
        self.T=T; steps=torch.arange(T+1,dtype=torch.float64)
        f=torch.cos((steps/T+s)/(1+s)*math.pi*0.5)**2
        self.alpha_bar=(f/f[0]).float()
        beta=1-self.alpha_bar[1:]/self.alpha_bar[:-1]
        self.beta=torch.clamp(beta,max=0.999).float(); self.alpha=1-self.beta
    def to(self,device):
        self.alpha_bar=self.alpha_bar.to(device); self.beta=self.beta.to(device)
        self.alpha=self.alpha.to(device); return self
    def add_noise(self,x0,noise,t):
        ab=self.alpha_bar[t].reshape(-1,1,1,1)
        return torch.sqrt(ab)*x0+torch.sqrt(1-ab)*noise
    def ddim_step(self,x_t,eps_pred,t,t_prev):
        ab_t=self.alpha_bar[t].reshape(-1,1,1,1); ab_p=self.alpha_bar[t_prev].reshape(-1,1,1,1)
        x0_pred=(x_t-torch.sqrt(1-ab_t)*eps_pred)/torch.sqrt(ab_t)
        return torch.sqrt(ab_p)*x0_pred+torch.sqrt(1-ab_p)*eps_pred

class SensorEmbedding(nn.Module):
    def __init__(self,srf_matrix,out_dim=64):
        super().__init__(); flat=torch.from_numpy(srf_matrix).float().flatten()
        self.register_buffer('srf_flat',flat)
        self.mlp=nn.Sequential(nn.Linear(flat.numel(),128),nn.GELU(),nn.Linear(128,out_dim))
    def forward(self): return self.mlp(self.srf_flat.unsqueeze(0)).squeeze(0)

class DiffusionNullFusion(nn.Module):
    def __init__(self,bands=128,msi=3,base_dim=48,scale=4,cond_dim=64,T=1000,num_heads=None,depths=None,window_size=8,use_checkpoint=False):
        super().__init__(); self.bands=bands; self.scale=scale; self.T=T
        srf=torch.from_numpy(chikusei_srf(bands)).float()
        self.register_buffer('srf',srf); self.register_buffer('srfinv',torch.linalg.pinv(srf))
        self.projector=RangeNullProjector(scale,cg_steps=8,ridge=1e-4)
        self.sensor_embed=SensorEmbedding(chikusei_srf(bands),cond_dim)
        self.schedule=CosineSchedule(T)
        self.cond_proj=nn.Sequential(nn.Conv2d(msi+bands,cond_dim,1),nn.AdaptiveAvgPool2d(1),nn.Flatten())
        self.time_mlp=nn.Sequential(nn.Linear(1,cond_dim),nn.GELU(),nn.Linear(cond_dim,cond_dim))
        self.unet=MultiScaleSwinUNet(in_ch=bands,base_dim=base_dim,num_heads=num_heads,cond_dim=cond_dim,window_size=window_size,depths=depths,use_checkpoint=use_checkpoint)
    def _conditioning(self,yH,yM,H_hr,W_hr):
        base=self.projector.pinv(yH,(H_hr,W_hr))
        cond=self.cond_proj(torch.cat([yM,base],1))+self.sensor_embed()
        return cond,base
    def forward(self,x_t,t,cond):
        with torch.autocast(device_type=x_t.device.type,enabled=x_t.is_cuda):
            out=self.unet(x_t,cond+self.time_mlp(t.float().unsqueeze(-1)))
        return out.float()
    @torch.no_grad()
    def inference(self,yH,yM,num_samples=5,ddim_steps=50):
        H_hr,W_hr=yM.shape[-2],yM.shape[-1]; cond,base=self._conditioning(yH,yM,H_hr,W_hr)
        samples=[]
        for _ in range(num_samples):
            x_t=torch.randn(1,self.bands,H_hr,W_hr,device=yH.device)
            ts=torch.linspace(self.T-1,0,ddim_steps,dtype=torch.long,device=yH.device)
            for i in range(len(ts)-1):
                eps=self(x_t,ts[i],cond); x_t=self.schedule.ddim_step(x_t,eps,ts[i],ts[i+1])
            samples.append(x_t)
        avg_null=self.projector.project_null(torch.mean(torch.stack(samples),0),(H_hr,W_hr))
        return {'out':base+avg_null,'base':base,'null':avg_null}

class ChikuseiDS(Dataset):
    def __init__(self,root,split='train',bands=128,scale=4,patch=64):
        self.split=split; self.bands=bands; self.scale=scale; self.patch=patch
        self.srf=chikusei_srf(bands); self.kernel=gaussian_kernel2d(9,1.2)
        mat_files=glob.glob(os.path.join(root,'**','*.mat'),recursive=True)
        hsi=[m for m in mat_files if 'Ground_Truth' not in os.path.basename(m) and 'gt' not in os.path.basename(m).lower()]
        if hsi: hsi.sort(key=lambda f:os.path.getsize(f),reverse=True); mat_path=hsi[0]
        elif mat_files: mat_files.sort(key=lambda f:os.path.getsize(f),reverse=True); mat_path=mat_files[0]
        else: raise FileNotFoundError(f'No .mat files under {root}')
        print(f'[Data] Loading: {mat_path}')
        try: data=loadmat(mat_path)
        except NotImplementedError:
            print('[Data] v7.3 mat file - using h5py'); data={}
            with h5py.File(mat_path,'r') as f:
                for key in f.keys():
                    if not key.startswith('__'):
                        val=np.array(f[key]).copy()
                        if val.ndim>=2: data[key]=val
        for key,val in data.items():
            if not key.startswith('__') and hasattr(val,'shape'):
                arr=np.array(val,dtype=np.float32)
                if arr.ndim==3 and min(arr.shape)>10:
                    if arr.shape[0]>arr.shape[-1]: arr=arr.transpose(2,0,1)
                    if arr.max()>1.0: arr=arr/arr.max()
                    self.cube=arr; break
        C,H,W=self.cube.shape; print(f'[Data] Cube: {C}b, {H}x{W}px')
        p=patch; coords=[(y,x) for y in range(0,H-p+1,p) for x in range(0,W-p+1,p)]
        random.seed(42); random.shuffle(coords); n=int(0.7*len(coords))
        self.patches=coords[:n] if split=='train' else coords[n:]
        print(f'[Data] {split}: {len(self.patches)} patches')
    def __len__(self): return len(self.patches)*(200 if self.split=='train' else 1)
    def _sim(self,gt):
        C,H,W=gt.shape; blurred=np.empty_like(gt)
        for c in range(C): blurred[c]=convolve(gt[c],self.kernel,mode='wrap')
        hr=H//self.scale; y0=(H-hr*self.scale)//2; x0=(W-hr*self.scale)//2
        lr=blurred[:,y0::self.scale,x0::self.scale].astype(np.float32)
        msi=np.einsum('chw,cm->mhw',gt,self.srf).astype(np.float32)
        return lr,np.clip(msi,0,1)
    def __getitem__(self,idx):
        y,x=self.patches[idx%len(self.patches)]; p=self.patch
        gt=self.cube[:,y:y+p,x:x+p].copy()
        if self.split=='train':
            if random.random()<0.5: gt=gt[:,:,::-1].copy()
            if random.random()<0.5: gt=gt[:,::-1,:].copy()
            if random.random()<0.5: gt=np.rot90(gt,random.randint(1,3),axes=(1,2)).copy()
            if random.random()<0.15: gt=(gt+np.random.randn(*gt.shape).astype(np.float32)*0.01).clip(0,1)
        lr,msi=self._sim(gt)
        return torch.from_numpy(gt),torch.from_numpy(lr),torch.from_numpy(msi)

def diffusion_loss(model,x0,cond,schedule,min_snr_gamma=5.0):
    B=x0.shape[0]; t=torch.randint(0,schedule.T,(B,),device=x0.device)
    noise=torch.randn_like(x0); x_t=schedule.add_noise(x0,noise,t)
    pred=model(x_t,t,cond)
    per_sample=F.mse_loss(pred,noise,reduction='none').flatten(1).mean(1)
    ab=schedule.alpha_bar[t].clamp(1e-5,1-1e-5)
    snr=ab/(1-ab)
    w=(snr.clamp(max=min_snr_gamma)/snr).detach()
    return (per_sample*w).mean()

def physics_loss(pred,yH,yM,model):
    return F.mse_loss(model.projector.D(pred),yH)+F.mse_loss(torch.einsum('bchw,cm->bmhw',pred,model.srf),yM)

def total_loss(model,gt,yH,yM,schedule,w_noise=1.0,w_phys=0.1,min_snr_gamma=5.0):
    H_hr,W_hr=yM.shape[-2],yM.shape[-1]; cond,base=model._conditioning(yH,yM,H_hr,W_hr)
    x0=model.projector.project_null(gt-base,(H_hr,W_hr))
    l_noise=diffusion_loss(model,x0,cond,schedule,min_snr_gamma)
    l_phys=physics_loss(base+x0,yH,yM,model)
    return w_noise*l_noise+w_phys*l_phys,l_noise,l_phys

def psnr_np(pred,gold):
    mse=np.mean((pred-gold)**2); return 100.0 if mse<1e-12 else -10*np.log10(mse)
def sam_np(pred,gold):
    p=pred.reshape(pred.shape[0],-1); g=gold.reshape(gold.shape[0],-1)
    p=p/(np.linalg.norm(p,axis=0,keepdims=True)+1e-8); g=g/(np.linalg.norm(g,axis=0,keepdims=True)+1e-8)
    return np.mean(np.arccos(np.clip((p*g).sum(0),-1,1)))*180/math.pi
def ssim_np(pred,gold):
    C1,C2=0.01**2,0.03**2; mu1=uniform_filter(pred,3,mode='reflect'); mu2=uniform_filter(gold,3,mode='reflect')
    s12=uniform_filter(pred*gold,3,mode='reflect')-mu1*mu2
    s1=uniform_filter(pred**2,3,mode='reflect')-mu1**2; s2=uniform_filter(gold**2,3,mode='reflect')-mu2**2
    return np.mean(((2*mu1*mu2+C1)*(2*s12+C2))/((mu1**2+mu2**2+C1)*(s1+s2+C2)+1e-8))
def ergas_np(pred,gold,scale=4):
    C=pred.shape[0]; e=sum(((pred-gold)**2)[c].mean()/(gold[c].mean()**2+1e-8) for c in range(C))
    return math.sqrt(e/C)*100*scale

class EMA:
    def __init__(self,m,d=0.999): self.d=d; self.s={k:v.detach().clone() for k,v in m.state_dict().items()}
    def update(self,m):
        with torch.no_grad():
            for k,v in m.state_dict().items():
                if v.dtype.is_floating_point and k in self.s: self.s[k].mul_(self.d).add_(v.detach(),alpha=1-self.d)
    def apply(self,m): m.load_state_dict(self.s,strict=False)
    def restore(self,m): self.s={k:v.detach().clone() for k,v in m.state_dict().items()}

def save_full_ckpt(path,model,ema,opt,scheduler,epoch,best_psnr,best_epoch):
    torch.save({'model':model.state_dict(),'ema':ema.s,'opt':opt.state_dict(),
                'scheduler':scheduler.state_dict(),'epoch':epoch,
                'best_psnr':best_psnr,'best_epoch':best_epoch},path)

def load_full_ckpt(path,model,ema,opt,scheduler,device):
    ck=torch.load(path,map_location=device,weights_only=False)
    model.load_state_dict(ck['model'])
    ema.s={k:v.to(device) for k,v in ck['ema'].items()}
    opt.load_state_dict(ck['opt'])
    scheduler.load_state_dict(ck['scheduler'])
    return ck['epoch'],ck['best_psnr'],ck['best_epoch']

print('Library OK')


In [ ]:
import train_diffusion_nullfusion as lib
import numpy as np
import torch, torch.nn as nn, random, time, os, json

GPU_OK = False
patch_size, batch_size, width = 64, 2, 48
use_checkpoint = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f'Device: {device} ({n_gpus} GPU{"s" if n_gpus != 1 else ""} visible)')

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    arch = f'sm_{p.major}{p.minor}'
    built = list(torch.cuda.get_arch_list())
    mem = p.total_memory / 1e9
    print(f'GPU0: {p.name} {mem:.1f} GB {arch}')
    print(f'Built for: {" ".join(built)}')
    GPU_OK = arch in built
    if not GPU_OK:
        print(f'\n*** GPU {arch} incompatible with this PyTorch -- falling back to CPU ***')
        print('For GPU: Settings -> Accelerator -> GPU T4 x2 -> Save -> Re-run')
        device = torch.device('cpu'); n_gpus = 0
    else:
        is_t4 = 'T4' in p.name
        if n_gpus >= 2 and is_t4:
            patch_size, batch_size, width = 72, 4, 52
            print(f'Note: {n_gpus}x T4 visible -- using GPU0 only.')
        elif is_t4:
            patch_size, batch_size, width = 72, 4, 52
        elif mem < 12:
            patch_size, batch_size, width = 64, 1, 48
        elif mem < 16:
            patch_size, batch_size, width = 64, 2, 48
        else:
            patch_size, batch_size, width = 80, 2, 56

model = lib.DiffusionNullFusion(bands=128, msi=3, base_dim=width, scale=4, cond_dim=64, T=1000,
                                 use_checkpoint=use_checkpoint).to(device)
nparams = sum(p.numel() for p in model.parameters())
use_amp = GPU_OK
print(f'Model: Diffusion-NullFusion -- {nparams/1e6:.2f}M params | AMP={use_amp} | grad_checkpoint={use_checkpoint}')
print(f'Config: patch={patch_size} batch={batch_size} width={width}')


In [ ]:
import glob, os

DATA_ROOT = None
search_dirs = ['/kaggle/input/chikusei', '/kaggle/input/mingliu123']
if os.path.isdir('/kaggle/input'):
    for d in os.listdir('/kaggle/input'):
        search_dirs.append(os.path.join('/kaggle/input', d))

for candidate in search_dirs:
    if not os.path.isdir(candidate): continue
    mats = glob.glob(os.path.join(candidate, '**', '*.mat'), recursive=True)
    hsi = [m for m in mats if 'Ground_Truth' not in os.path.basename(m) and 'gt' not in os.path.basename(m).lower()]
    if hsi:
        hsi.sort(key=lambda f: os.path.getsize(f), reverse=True)
        DATA_ROOT = candidate
        print(f'[Data] Found: {hsi[0]}')
        break

if DATA_ROOT is None:
    print('Listing /kaggle/input/:')
    if os.path.isdir('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            depth = root.replace('/kaggle/input', '').count(os.sep)
            if depth > 2: continue
            indent = '  ' * depth
            print(f'{indent}{os.path.basename(root)}/')
            for f in files[:3]:
                print(f'{indent}  {f} ({os.path.getsize(os.path.join(root,f))/1e6:.1f} MB)')
    raise FileNotFoundError('Cannot find Chikusei .mat file.')

train_ds = lib.ChikuseiDS(DATA_ROOT, 'train', 128, 4, patch_size)
test_ds = lib.ChikuseiDS(DATA_ROOT, 'test', 128, 4, patch_size)

grad_accum = 4; lr = 2e-4; epochs = 5000; steps_per_epoch = 200; eval_every = 100
time_budget_h = 8.5
T_ddpm = 1000; ddim_steps = 50; num_samples = 5
eval_subset = 60
min_snr_gamma = 5.0

opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4, betas=(0.9, 0.999))
ema = lib.EMA(model, 0.999)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=2000, T_mult=2, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

schedule = model.schedule.to(device)
save_dir = '/kaggle/working/diffusion_nullfusion'
resume_ckpt = os.path.join(save_dir, 'resume.pth')

start_epoch = 1
best_psnr = float('-inf'); best_epoch = 0
if os.path.exists(resume_ckpt):
    start_epoch, best_psnr, best_epoch = lib.load_full_ckpt(resume_ckpt, model, ema, opt, scheduler, device)
    start_epoch += 1
    print(f'[Resume] Loaded checkpoint -- continuing from epoch {start_epoch} '
          f'(best PSNR so far: {best_psnr:.4f} @ epoch {best_epoch})')

T0 = time.time(); LIMIT = time_budget_h * 3600

targets = {
    'CoFusion (2026)': 50.67,
    'SMGU-Net (2025)': 49.83,
    'PSRT (2023)': 47.99,
    'RAMoE': 48.10,
}
STRETCH_TARGET = 53.0

print(f'\nTraining: {epochs} epochs, {time_budget_h}h budget (this session)')
print(f'Diffusion T={T_ddpm}, DDIM steps={ddim_steps}, samples={num_samples}')
print(f'Effective batch: {batch_size * grad_accum}, Steps/epoch: {steps_per_epoch}')
print('-' * 70)


In [ ]:
import numpy as np

for epoch in range(start_epoch, epochs + 1):
    if (time.time() - T0) > LIMIT:
        print(f'\n[TIME BUDGET at epoch {epoch}]')
        lib.save_full_ckpt(resume_ckpt, model, ema, opt, scheduler, epoch - 1, best_psnr, best_epoch)
        print('[Resume] Saved checkpoint -- re-run the notebook next session to continue training.')
        break

    model.train(); total = 0; t0 = time.time(); opt.zero_grad()
    for step in range(steps_per_epoch):
        gts, lhs, mss = [], [], []
        for _ in range(batch_size):
            g, l, m = train_ds[random.randrange(len(train_ds))]
            gts.append(g); lhs.append(l); mss.append(m)
        gt = torch.stack(gts, 0).to(device)
        yH = torch.stack(lhs, 0).to(device)
        yM = torch.stack(mss, 0).to(device)
        loss, l_noise, l_phys = lib.total_loss(model, gt, yH, yM, schedule, min_snr_gamma=min_snr_gamma)
        scaler.scale(loss / grad_accum).backward()
        if (step + 1) % grad_accum == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(); ema.update(model)
        total += loss.item()
    scheduler.step(); dt = time.time() - t0; avg = total / steps_per_epoch

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:5d}/{epochs} | Loss {avg:.5f} | LR {scheduler.get_last_lr()[0]:.2e} | {dt:.1f}s')

    if epoch % eval_every == 0 or epoch == epochs:
        raw_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        ema.apply(model); model.eval()
        ps, ss, sa, er = [], [], [], []
        eval_idx = random.sample(range(len(test_ds)), min(eval_subset, len(test_ds)))
        with torch.no_grad():
            for i in eval_idx:
                g, l, m = test_ds[i]
                out = model.inference(l.unsqueeze(0).to(device), m.unsqueeze(0).to(device), num_samples=1, ddim_steps=20)
                pred_np = out['out'][0].cpu().numpy(); gt_np = g.numpy()
                ps.append(lib.psnr_np(pred_np, gt_np)); ss.append(lib.ssim_np(pred_np, gt_np))
                sa.append(lib.sam_np(pred_np, gt_np)); er.append(lib.ergas_np(pred_np, gt_np, 4))
        mp, ms_, ma, me = float(np.mean(ps)), float(np.mean(ss)), float(np.mean(sa)), float(np.mean(er))
        marker = ''
        if mp > best_psnr:
            best_psnr = mp; best_epoch = epoch
            torch.save(model.state_dict(), os.path.join(save_dir, 'best.pth'))
            marker = ' [BEST]'
        print(f'  >>> Test@{epoch} (n={len(eval_idx)} subset): PSNR {mp:.4f} | SSIM {ms_:.4f} | SAM {ma:.3f} | ERGAS {me:.3f}{marker}')
        for name, target in targets.items():
            d = mp - target; m_ = ' <<< BEAT' if d > 0 else ''
            print(f'    vs {name:15s}: delta={d:+.2f} dB{m_}')
        model.load_state_dict(raw_state)
        model.train()
        lib.save_full_ckpt(resume_ckpt, model, ema, opt, scheduler, epoch, best_psnr, best_epoch)


In [ ]:
import numpy as np

print('\n' + '=' * 70)
print('FINAL EVALUATION: 5 samples x 50 DDIM steps, FULL test set')
print('=' * 70)

ckpt = os.path.join(save_dir, 'best.pth')
if os.path.exists(ckpt): model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval(); ps, ss, sa, er = [], [], [], []
with torch.no_grad():
    for i in range(len(test_ds)):
        g, l, m = test_ds[i]
        out = model.inference(l.unsqueeze(0).to(device), m.unsqueeze(0).to(device), num_samples=num_samples, ddim_steps=ddim_steps)
        pred_np = out['out'][0].cpu().numpy(); gt_np = g.numpy()
        ps.append(lib.psnr_np(pred_np, gt_np)); ss.append(lib.ssim_np(pred_np, gt_np))
        sa.append(lib.sam_np(pred_np, gt_np)); er.append(lib.ergas_np(pred_np, gt_np, 4))
final = {'psnr': float(np.mean(ps)), 'ssim': float(np.mean(ss)), 'sam': float(np.mean(sa)), 'ergas': float(np.mean(er))}
print(f'\nFINAL (epoch {best_epoch}): PSNR {final["psnr"]:.4f} | SSIM {final["ssim"]:.4f} | SAM {final["sam"]:.3f} | ERGAS {final["ergas"]:.3f}')
print('=' * 70)
for name, target in targets.items():
    d = final['psnr'] - target; m_ = ' <<< BEAT' if d > 0 else ''
    print(f'  {name:25s} Target {target:6.2f} | Ours delta={d:+.2f} dB{m_}')
d_stretch = final['psnr'] - STRETCH_TARGET
print(f'  {"Stretch target":25s} Target {STRETCH_TARGET:6.2f} | Ours delta={d_stretch:+.2f} dB' + (' <<< BEAT' if d_stretch > 0 else ''))
with open('/kaggle/working/diffusion_nullfusion/results.json', 'w') as f:
    json.dump({'dataset': 'Chikusei', 'bands': 128, 'params_M': nparams/1e6, 'best_epoch': best_epoch,
               'T': T_ddpm, 'ddim_steps': ddim_steps, 'num_samples': num_samples,
               'targets': targets, 'stretch_target': STRETCH_TARGET, 'final': final}, f, indent=2)
print('\nSaved results.json')
